# Final End-To-End Validation And Submission

This notebook is the final end-to-end validation layer. It loads the submission dataset, runs the production `InferenceEngine` without bypassing stage boundaries, validates per-problem stage artifacts plus final answer integrity, writes inspectable run diagnostics, and emits a strict `submission.csv` only after every problem produces a valid integer answer in `[0, 999]`.


In [ ]:
from __future__ import annotations

import json
import shutil
import time
import traceback
from collections import Counter
from dataclasses import asdict, is_dataclass
from pathlib import Path
from typing import Any, Mapping

from src.common.constants import ANSWER_MAX, ANSWER_MIN
from src.online.inference_engine import InferenceEngine, InferenceEngineConfig
from src.online.kaggle_runner import read_kaggle_input_csv, write_submission_csv
from src.online.submission_formatter import SubmissionRow

SEED = 1337
OVERWRITE_OUTPUTS = True

ROOT = Path.cwd()
INPUT_CSV_CANDIDATES = (
    ROOT / 'data' / 'raw' / 'aimo3_test.csv',
    ROOT / 'data' / 'raw' / 'test.csv',
    ROOT / 'input' / 'aimo3_test.csv',
    ROOT / 'input' / 'test.csv',
)
INDEX_PATH = ROOT / 'data' / 'interim' / 'retrieval_index'
RUN_ID = f'submission_validation_seed{SEED}'
OUT_DIR = ROOT / 'artifacts' / 'submission_validation' / RUN_ID
RUN_RECORDS_PATH = OUT_DIR / 'submission_run_records.json'
SUMMARY_PATH = OUT_DIR / 'submission_validation_summary.json'
ARCHIVE_SUBMISSION_PATH = OUT_DIR / 'submission.csv'
SUBMISSION_PATH = ROOT / 'submission.csv'

EXPECTED_STAGE_SEQUENCE = (
    'parse',
    'route',
    'budget',
    'state_init',
    'retrieval',
    'branch_controller',
    'aggregation',
    'submission',
)


def need(condition, message):
    if not condition:
        raise AssertionError(message)


def sj(value):
    return json.dumps(value, ensure_ascii=True, sort_keys=True, default=str)


def text(value):
    return ' '.join(str(value or '').split())


def fnum(value, default=0.0):
    try:
        return float(value)
    except Exception:
        return float(default)


def model_to_data(value):
    if value is None:
        return None
    if hasattr(value, 'model_dump'):
        return value.model_dump(mode='json')
    if is_dataclass(value):
        return asdict(value)
    if isinstance(value, Mapping):
        return dict(value)
    if isinstance(value, (list, tuple)):
        return [model_to_data(item) for item in value]
    return value


def prepare_output_dir(path, overwrite):
    if path.exists():
        if not overwrite:
            raise FileExistsError(f'Output directory already exists: {path}')
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def resolve_input_csv(candidates):
    inspected = []
    for path in candidates:
        exists = path.exists()
        size = path.stat().st_size if exists else 0
        inspected.append({'path': str(path), 'exists': exists, 'size_bytes': int(size)})
        if exists and size > 0:
            return path, inspected
    raise FileNotFoundError(
        'No non-empty submission input CSV found in configured candidates: '
        + sj(inspected)
    )


def normalize_submission_row(row):
    need(row is not None, 'Inference engine did not emit a submission row.')
    if isinstance(row, SubmissionRow):
        normalized = row
    elif hasattr(row, 'model_dump'):
        normalized = SubmissionRow.model_validate(row.model_dump(mode='json'))
    elif isinstance(row, Mapping):
        normalized = SubmissionRow.model_validate(dict(row))
    else:
        raise TypeError(f'Unsupported submission row type: {type(row)!r}')
    need(isinstance(normalized.answer, int) and not isinstance(normalized.answer, bool), f'Submission answer must be an integer, got {type(normalized.answer)!r}.')
    need(ANSWER_MIN <= normalized.answer <= ANSWER_MAX, f'Submission answer out of range for {normalized.id}: {normalized.answer}')
    need(text(normalized.id), 'Submission row id is empty.')
    return normalized


def validate_stage_sequence(result):
    stage_names = [str(getattr(item.stage, 'value', item.stage)) for item in result.stage_records]
    need(stage_names, f'Stage records missing for {result.problem_id}.')
    missing = [name for name in EXPECTED_STAGE_SEQUENCE if name not in stage_names]
    need(not missing, f'Missing required stages for {result.problem_id}: {missing}; observed={stage_names}')
    ordered_positions = [stage_names.index(name) for name in EXPECTED_STAGE_SEQUENCE]
    need(ordered_positions == sorted(ordered_positions), f'Stage order mismatch for {result.problem_id}: {stage_names}')
    failed = [item for item in result.stage_records if not bool(item.ok)]
    need(not failed, f'Failed stage records for {result.problem_id}: {[str(getattr(item.stage, "value", item.stage)) for item in failed]}')
    return stage_names


def validate_branch_result(result, submission_row):
    branch_result = result.branch_result
    need(branch_result is not None, f'Branch controller result missing for {result.problem_id}.')
    need(result.final_selection is not None, f'Final selection missing for {result.problem_id}.')
    need(result.final_prediction is not None, f'Final prediction missing for {result.problem_id}.')
    need(branch_result.final_prediction.problem_id == result.problem_id, f'Branch final prediction problem_id mismatch for {result.problem_id}.')
    need(int(branch_result.final_prediction.final_answer) == int(submission_row.answer), f'Final prediction and submission answer diverged for {result.problem_id}.')
    branches = list(branch_result.all_branches)
    survivors = list(branch_result.surviving_branches)
    need(branches, f'No branches generated for {result.problem_id}.')
    need(survivors, f'No surviving branches available for {result.problem_id}.')
    need(branch_result.candidate_clusters, f'No candidate clusters produced for {result.problem_id}.')

    step_kind_counts = Counter()
    symbolic_evidence_total = 0
    verifier_evidence_total = 0
    retrieval_evidence_total = 0
    candidate_count = 0
    failure_type_counts = Counter()

    for branch in branches:
        need(branch.problem_id == result.problem_id, f'Branch problem_id mismatch inside branch controller result for {result.problem_id}.')
        summary = branch.build_summary()
        steps = list(branch.active_steps())
        symbolic = list(branch.active_symbolic_evidence())
        verifier = list(branch.active_verifier_evidence())
        retrieval = list(branch.active_retrieval_evidence())
        need(summary.current_step_count == len(steps), f'Active step count mismatch for branch {branch.branch_id}.')
        need(summary.symbolic_count == len(symbolic), f'Symbolic evidence count mismatch for branch {branch.branch_id}.')
        need(summary.verifier_count == len(verifier), f'Verifier evidence count mismatch for branch {branch.branch_id}.')
        need(summary.retrieval_count == len(retrieval), f'Retrieval evidence count mismatch for branch {branch.branch_id}.')
        need(branch.score_breakdown.exact_symbolic_check >= 0.0, f'Invalid symbolic score for branch {branch.branch_id}.')
        need(branch.score_breakdown.verifier_probability >= 0.0, f'Invalid verifier score for branch {branch.branch_id}.')
        if branch.current_candidate() is not None:
            candidate_count += 1
        raw_failure = branch.metadata.get('failure_type')
        if raw_failure is not None:
            failure_type_counts[str(getattr(raw_failure, 'value', raw_failure))] += 1
        symbolic_evidence_total += len(symbolic)
        verifier_evidence_total += len(verifier)
        retrieval_evidence_total += len(retrieval)
        for step in steps:
            step_kind_counts[str(getattr(step.kind, 'value', step.kind))] += 1

    need(retrieval_evidence_total > 0 or len(result.retrieved_traces) == 0, f'Retrieval stage succeeded but branches contain no retrieval evidence for {result.problem_id}.')
    need(symbolic_evidence_total > 0, f'No symbolic evidence recorded for {result.problem_id}.')
    need(verifier_evidence_total > 0, f'No verifier evidence recorded for {result.problem_id}.')
    need(candidate_count > 0, f'No branch candidate answers recorded for {result.problem_id}.')

    return {
        'branch_count': len(branches),
        'surviving_branch_count': len(survivors),
        'candidate_cluster_count': len(branch_result.candidate_clusters),
        'retrieved_trace_count': len(result.retrieved_traces),
        'retrieval_evidence_count': retrieval_evidence_total,
        'symbolic_evidence_count': symbolic_evidence_total,
        'verifier_evidence_count': verifier_evidence_total,
        'candidate_branch_count': candidate_count,
        'stopped_reason': str(branch_result.stopped_reason),
        'step_kind_counts': dict(sorted(step_kind_counts.items())),
        'failure_type_counts': dict(sorted(failure_type_counts.items())),
    }


In [ ]:
prepare_output_dir(OUT_DIR, OVERWRITE_OUTPUTS)
if SUBMISSION_PATH.exists() and not OVERWRITE_OUTPUTS:
    raise FileExistsError(f'submission.csv already exists: {SUBMISSION_PATH}')

input_csv_path, inspected_candidates = resolve_input_csv(INPUT_CSV_CANDIDATES)
records = read_kaggle_input_csv(input_csv_path, validate_columns=True)
need(records, f'Input CSV has no records: {input_csv_path}')

engine = InferenceEngine(
    config=InferenceEngineConfig(
        deterministic_seed=SEED,
        enable_debug_artifacts=True,
        include_submission_row=True,
        strict_stage_failures=True,
        retrieval_index_path=str(INDEX_PATH),
        include_trace_hint_text=True,
        post_branch_budget_adaptation=True,
    )
)

submission_rows = []
run_records = []
status_counts = Counter()
failed_problem_ids = []

for index, record in enumerate(records):
    started = time.perf_counter()
    problem_id = str(record.id)
    remaining = len(records) - index
    try:
        result = engine.solve_problem(record.problem, problem_id=problem_id, remaining_problems=remaining)
        elapsed = time.perf_counter() - started
        status = str(getattr(result.status, 'value', result.status))
        status_counts[status] += 1
        stage_names = validate_stage_sequence(result)
        submission_row = normalize_submission_row(result.submission_row)
        branch_stats = validate_branch_result(result, submission_row)
        need(result.ok, f'Inference engine reported non-success status for {problem_id}: {status}')
        need(result.failure is None, f'Inference engine reported failure for {problem_id}: {model_to_data(result.failure)}')
        need(result.route is not None, f'RouteDecision missing for {problem_id}.')
        need(result.parsed_problem is not None, f'ParsedProblem missing for {problem_id}.')
        need(result.state_init is not None, f'State initialization missing for {problem_id}.')
        submission_rows.append(submission_row)
        run_records.append(
            {
                'problem_id': problem_id,
                'status': status,
                'ok': True,
                'elapsed_sec': round(elapsed, 6),
                'answer': int(submission_row.answer),
                'stage_names': stage_names,
                'route_difficulty': str(getattr(getattr(result.route, 'difficulty', None), 'value', getattr(result.route, 'difficulty', ''))),
                'route_difficulty_score': fnum(getattr(result.route, 'difficulty_score', 0.0)),
                'route_retrieval_depth': int(getattr(result.route, 'retrieval_depth', 0)),
                'route_branch_budget': int(getattr(result.route, 'branch_budget', 0)),
                'route_verifier_mode': str(getattr(result.route, 'verifier_mode', '')),
                'route_use_symbolic': bool(getattr(result.route, 'use_symbolic', False)),
                'route_use_retrieval': bool(getattr(result.route, 'use_retrieval', False)),
                'final_confidence': fnum(getattr(result.final_prediction, 'confidence', 0.0)),
                'final_method': str(getattr(result.final_prediction, 'method_used', '')),
                'debug_artifact_available': result.debug_artifact is not None,
                'failure': None,
                'branch_stats': branch_stats,
            }
        )
    except Exception as exc:
        elapsed = time.perf_counter() - started
        status_counts['exception'] += 1
        failed_problem_ids.append(problem_id)
        run_records.append(
            {
                'problem_id': problem_id,
                'status': 'exception',
                'ok': False,
                'elapsed_sec': round(elapsed, 6),
                'answer': None,
                'stage_names': [],
                'route_difficulty': '',
                'route_difficulty_score': 0.0,
                'route_retrieval_depth': 0,
                'route_branch_budget': 0,
                'route_verifier_mode': '',
                'route_use_symbolic': False,
                'route_use_retrieval': False,
                'final_confidence': 0.0,
                'final_method': '',
                'debug_artifact_available': False,
                'failure': {
                    'exception_type': type(exc).__name__,
                    'message': str(exc),
                    'traceback': traceback.format_exc(limit=8),
                },
                'branch_stats': {},
            }
        )

run_records_payload = {
    'run_id': RUN_ID,
    'seed': SEED,
    'input_csv_path': str(input_csv_path),
    'records': run_records,
}
RUN_RECORDS_PATH.write_text(json.dumps(run_records_payload, indent=2, ensure_ascii=True, sort_keys=True), encoding='utf-8')

answers = [row.answer for row in submission_rows]
duplicate_ids = [problem_id for problem_id, count in Counter(row.id for row in submission_rows).items() if count > 1]
missing_answer_problem_ids = [entry['problem_id'] for entry in run_records if entry['answer'] is None]
summary_payload = {
    'run_id': RUN_ID,
    'seed': SEED,
    'input_csv_path': str(input_csv_path),
    'input_candidate_scan': inspected_candidates,
    'problem_count': len(records),
    'solved_count': len(submission_rows),
    'status_counts': dict(sorted(status_counts.items())),
    'failed_problem_ids': failed_problem_ids,
    'missing_answer_problem_ids': missing_answer_problem_ids,
    'duplicate_problem_ids': sorted(duplicate_ids),
    'answer_min': min(answers) if answers else None,
    'answer_max': max(answers) if answers else None,
    'mean_elapsed_sec': round(sum(entry['elapsed_sec'] for entry in run_records) / max(1, len(run_records)), 6),
}
SUMMARY_PATH.write_text(json.dumps(summary_payload, indent=2, ensure_ascii=True, sort_keys=True), encoding='utf-8')

need(len(submission_rows) == len(records), f'All problems must produce answers. solved={len(submission_rows)} expected={len(records)} failed={failed_problem_ids}')
need(not failed_problem_ids, f'Inference failed for problems: {failed_problem_ids}')
need(not missing_answer_problem_ids, f'Missing answers detected: {missing_answer_problem_ids}')
need(not duplicate_ids, f'Duplicate submission ids detected: {duplicate_ids}')
need(all(isinstance(answer, int) and not isinstance(answer, bool) for answer in answers), 'Submission answers must all be integers.')
need(all(ANSWER_MIN <= answer <= ANSWER_MAX for answer in answers), f'Submission answers must stay in [{ANSWER_MIN}, {ANSWER_MAX}].')

write_submission_csv(submission_rows, SUBMISSION_PATH)
ARCHIVE_SUBMISSION_PATH.write_text(SUBMISSION_PATH.read_text(encoding='utf-8'), encoding='utf-8')

summary_payload = json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))
summary_payload.update(
    {
        'submission_path': str(SUBMISSION_PATH),
        'archive_submission_path': str(ARCHIVE_SUBMISSION_PATH),
        'submission_emitted': True,
    }
)
SUMMARY_PATH.write_text(json.dumps(summary_payload, indent=2, ensure_ascii=True, sort_keys=True), encoding='utf-8')
summary_payload


In [ ]:
summary_payload = json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))
submission_preview = SUBMISSION_PATH.read_text(encoding='utf-8').splitlines()[:10]

final_report = {
    'input_csv_path': summary_payload['input_csv_path'],
    'problem_count': summary_payload['problem_count'],
    'solved_count': summary_payload['solved_count'],
    'status_counts': summary_payload['status_counts'],
    'answer_range': [summary_payload['answer_min'], summary_payload['answer_max']],
    'submission_path': summary_payload['submission_path'],
    'archive_submission_path': summary_payload['archive_submission_path'],
    'run_records_path': str(RUN_RECORDS_PATH),
}

print(json.dumps(final_report, indent=2, ensure_ascii=True, sort_keys=True))
submission_preview
